In [1]:
# ─────────────────────────────────────────────────────────────
# DIAGNOSTIC — trend anchor sweep
# Goal: locate GLA M/F trend + LSDB yearly trend, dump shapes.
# Sources:
#   GLA Basis    → "Male trend factor" + "Female trend factor"
#                  (salary bands × years 2015-2023)
#   LSDB Basis   → "Year" anchor (single value per year, ~2015-2026)
# ─────────────────────────────────────────────────────────────
from pathlib import Path
from openpyxl import load_workbook

BASE   = Path(r"J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine")
SOURCE = BASE / "Basis and Assumptions.xlsx"
assert SOURCE.exists(), f"Source not found: {SOURCE}"

wb = load_workbook(SOURCE, data_only=True, read_only=False)

def find_anchors(ws, text):
    hits = []
    target = text.strip()
    for row in ws.iter_rows():
        for cell in row:
            v = cell.value
            if isinstance(v, str) and v.strip() == target:
                hits.append((cell.row, cell.column))
    return hits

def dump_block(ws, r0, c0, n_rows=14, n_cols=11, label=""):
    print(f"\n── {label}  anchor at (row={r0}, col={c0})  sheet='{ws.title}' ──")
    print(f"{'row':>4} | " + " | ".join(f"c{c0+i:>2}" for i in range(n_cols)))
    print("-" * (7 + n_cols * 9))
    for dr in range(n_rows):
        r = r0 + dr
        vals = []
        for dc in range(n_cols):
            v = ws.cell(row=r, column=c0 + dc).value
            if v is None:
                vals.append("·")
            elif isinstance(v, float):
                vals.append(f"{v:.4g}")
            else:
                vals.append(str(v)[:7])
        print(f"{r:>4} | " + " | ".join(f"{s:>7}" for s in vals))

# ═════════════════════════════════════════════════════════════
# 1) GLA Basis — Male + Female trend factor
# ═════════════════════════════════════════════════════════════
ws_gla = wb["GLA Basis"]

for label in ["Male trend factor", "Female trend factor"]:
    hits = find_anchors(ws_gla, label)
    print(f"\nGLA Basis — {label!r} hits: {hits}")
    for i, (r, c) in enumerate(hits, 1):
        # Dump wide — need to see the year headers + all salary bands
        dump_block(ws_gla, r, c, n_rows=8, n_cols=11, label=f"{label} #{i}")

# Also probe the year header row — should be right below/near anchor
print("\n[GLA Basis] variant probe:")
for v in ["Salary", "2015", "2016", "2023"]:
    hits = find_anchors(ws_gla, v)
    if hits:
        print(f"  {v!r:>10} -> {hits[:5]}{' …' if len(hits) > 5 else ''}  (total={len(hits)})")

# ═════════════════════════════════════════════════════════════
# 2) LSDB Basis — yearly trend
# ═════════════════════════════════════════════════════════════
ws_lsdb = wb["LSDB Basis"]

# Handover Appendix A: anchor "Year"
print("\n\n═══ LSDB Basis ═══")
for label in ["Year", "Yearly loading", "Yearly trend", "Trend"]:
    hits = find_anchors(ws_lsdb, label)
    if hits:
        print(f"\nLSDB Basis — {label!r} hits: {hits}")
        for i, (r, c) in enumerate(hits, 1):
            dump_block(ws_lsdb, r, c - 1, n_rows=16, n_cols=4, label=f"{label} #{i}")
    else:
        print(f"\nLSDB Basis — {label!r}: no hits")

# Broad probe on LSDB
print("\n[LSDB Basis] variant probe:")
for v in ["2015", "2016", "2025", "2026"]:
    hits = find_anchors(ws_lsdb, v)
    if hits:
        print(f"  {v!r:>10} -> {hits[:5]}{' …' if len(hits) > 5 else ''}  (total={len(hits)})")



GLA Basis — 'Male trend factor' hits: [(17, 10)]

── Male trend factor #1  anchor at (row=17, col=10)  sheet='GLA Basis' ──
 row | c10 | c11 | c12 | c13 | c14 | c15 | c16 | c17 | c18 | c19 | c20
----------------------------------------------------------------------------------------------------------
  17 | Male tr |       · |       · |       · |       · |       · |       · |       · |       · |       · |       ·
  18 |  Salary |    2015 |    2016 |    2017 |    2018 |    2019 |    2020 |    2021 |    2022 |    2023 |       ·
  19 |       0 |    1.08 |    1.06 |    1.04 |    1.02 |       1 |    0.98 |    0.96 |    0.94 |    0.92 |       ·
  20 |  226000 |    1.04 |    1.03 |    1.02 |    1.01 |       1 |    0.99 |    0.98 |    0.97 |    0.96 |       ·
  21 |  504000 |     0.8 |    0.85 |     0.9 |    0.95 |       1 |    1.05 |     1.1 |    1.15 |     1.2 |       ·
  22 |       · |       · |       · |       · |       · |       · |       · |       · |       · |       · |       ·
  23 | 

In [2]:
# ═══════════════════════════════════════════════════════════════
# build_trend.ipynb — Cell 1: Setup
# ═══════════════════════════════════════════════════════════════
# Builds the 'trend' sheet of the Basis Master.
# Schema: product | gender | salary_from | year | factor
# Products:
#   gla   → M + F, 3 salary bands each × 9 years (2015-2023) = 54 rows
#   lsdb  → both,  no bands, 12 years (2015-2026)            = 12 rows
# Expected total rows: 66
# ═══════════════════════════════════════════════════════════════

import csv
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import polars as pl
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

# ── Paths ──────────────────────────────────────────────────────
BASE          = Path(r"J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine")
SOURCE        = BASE / "Basis and Assumptions.xlsx"
MASTER_XLSX   = BASE / "basis_master" / "basis_master_v2026_07.xlsx"
SNAPSHOT_DIR  = BASE / "basis_snapshots" / "basis_v2026_07"
BUILD_LOG     = SNAPSHOT_DIR / "build_log.csv"

# ── Sheet identity ─────────────────────────────────────────────
SHEET_NAME    = "trend"
PARQUET_PATH  = SNAPSHOT_DIR / f"{SHEET_NAME}.parquet"
BASIS_VERSION = "B2026_07"
BUILT_BY      = "Mohamed Hussein Gadiwala"

# ── Ensure output dirs exist ───────────────────────────────────
SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
MASTER_XLSX.parent.mkdir(parents=True, exist_ok=True)

# ── Source guard ───────────────────────────────────────────────
assert SOURCE.exists(),       f"Source workbook not found: {SOURCE}"
assert SNAPSHOT_DIR.exists(), f"Snapshot dir missing:      {SNAPSHOT_DIR}"

# ── Master xlsx open-lock guard ────────────────────────────────
if MASTER_XLSX.exists():
    try:
        with open(MASTER_XLSX, "a"):
            pass
    except PermissionError:
        raise PermissionError(
            f"'{MASTER_XLSX.name}' is open in Excel. Close it and re-run."
        )

# ── Sanity print ───────────────────────────────────────────────
print("✅ Cell 1 — Setup complete")
print(f"   Source        : {SOURCE.name}")
print(f"   Master xlsx   : {MASTER_XLSX.name}  (exists: {MASTER_XLSX.exists()})")
print(f"   Snapshot dir  : {SNAPSHOT_DIR}")
print(f"   Target sheet  : {SHEET_NAME}")
print(f"   Basis version : {BASIS_VERSION}")

✅ Cell 1 — Setup complete
   Source        : Basis and Assumptions.xlsx
   Master xlsx   : basis_master_v2026_07.xlsx  (exists: True)
   Snapshot dir  : J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine\basis_snapshots\basis_v2026_07
   Target sheet  : trend
   Basis version : B2026_07


In [3]:
# ═══════════════════════════════════════════════════════════════
# build_trend.ipynb — Cell 2: Helpers
# ═══════════════════════════════════════════════════════════════

def find_anchors(ws, text):
    """Return (row, col) tuples for cells whose stripped str value == text."""
    hits = []
    target = text.strip()
    for row in ws.iter_rows():
        for cell in row:
            v = cell.value
            if isinstance(v, str) and v.strip() == target:
                hits.append((cell.row, cell.column))
    return hits


def as_str(v):
    if v is None:
        return None
    s = str(v).strip()
    return s if s else None


def as_float(v):
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return float(v)
    s = str(v).strip()
    if not s:
        return None
    try:
        return float(s)
    except ValueError:
        return None


def as_int(v):
    if v is None:
        return None
    if isinstance(v, int) and not isinstance(v, bool):
        return v
    if isinstance(v, float):
        return int(v) if v.is_integer() else None
    s = str(v).strip()
    if not s:
        return None
    try:
        return int(s)
    except ValueError:
        try:
            f = float(s)
            return int(f) if f.is_integer() else None
        except ValueError:
            return None


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()


def utc_now_iso():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


print("✅ Cell 2 — Helpers loaded")

✅ Cell 2 — Helpers loaded


In [4]:
# ═══════════════════════════════════════════════════════════════
# build_trend.ipynb — Cell 3: SPECS
# ═══════════════════════════════════════════════════════════════
# Two shapes:
#
# SHAPE A — GLA matrix (M and F):
#   Anchor    : "Male trend factor" / "Female trend factor" at col C
#   Row 0     : anchor (title)
#   Row 1     : Salary | 2015 | 2016 | ... | 2023   (year headers at col C+1..C+9)
#   Row 2..N  : <salary_from> | <factor 2015> | ... | <factor 2023>
#   Walk down salary_from column until blank.
#
# SHAPE B — LSDB two-column list:
#   Anchor    : "Year" at (18, 5) — this is a column header, not a title
#   Row 0     : Year | Loading
#   Row 1..N  : <year> | <factor>
#   Walk down year column until blank.
# ═══════════════════════════════════════════════════════════════

SPECS = [
    # ── SHAPE A ────────────────────────────────────────────────
    {
        "shape":         "gla_matrix",
        "product":       "gla",
        "gender":        "M",
        "sheet":         "GLA Basis",
        "anchor_text":   "Male trend factor",
        "anchor_occ":    1,
        "expected_bands": 3,       # 0, 226k, 504k
        "expected_years": 9,       # 2015..2023
        "expected_rows": 27,
    },
    {
        "shape":         "gla_matrix",
        "product":       "gla",
        "gender":        "F",
        "sheet":         "GLA Basis",
        "anchor_text":   "Female trend factor",
        "anchor_occ":    1,
        "expected_bands": 3,       # 0, 214k, 441k
        "expected_years": 9,
        "expected_rows": 27,
    },
    # ── SHAPE B ────────────────────────────────────────────────
    {
        "shape":         "lsdb_list",
        "product":       "lsdb",
        "gender":        "both",
        "sheet":         "LSDB Basis",
        "anchor_text":   "Year",
        "anchor_occ":    1,
        "expected_years": 12,      # 2015..2026
        "expected_rows": 12,
    },
]

# ── Load workbook once ─────────────────────────────────────────
wb = load_workbook(SOURCE, data_only=True, read_only=False)

# ── Validate every anchor resolves ─────────────────────────────
print("🔍 Anchor validation:")
for spec in SPECS:
    ws = wb[spec["sheet"]]
    hits = find_anchors(ws, spec["anchor_text"])
    occ = spec["anchor_occ"]
    assert len(hits) >= occ, (
        f"[{spec['product']}/{spec.get('gender', '?')}] expected ≥{occ} "
        f"'{spec['anchor_text']}' hits on '{spec['sheet']}', got {len(hits)}"
    )
    r, c = hits[occ - 1]
    tag = f"{spec['product']}/{spec['gender']}"
    print(f"   {tag:<10} → {spec['sheet']:<14} "
          f"[{spec['shape']:<11}] anchor #{occ} at (row={r}, col={c})")

print("\n✅ Cell 3 — SPECS loaded, all anchors resolved")

🔍 Anchor validation:
   gla/M      → GLA Basis      [gla_matrix ] anchor #1 at (row=17, col=10)
   gla/F      → GLA Basis      [gla_matrix ] anchor #1 at (row=23, col=10)
   lsdb/both  → LSDB Basis     [lsdb_list  ] anchor #1 at (row=18, col=5)

✅ Cell 3 — SPECS loaded, all anchors resolved


In [5]:
# ═══════════════════════════════════════════════════════════════
# build_trend.ipynb — Cell 4: Extract
# ═══════════════════════════════════════════════════════════════
# Two extraction strategies driven by spec["shape"].
# ═══════════════════════════════════════════════════════════════

all_rows = []
MAX_WALK = 100  # safety cap


def extract_gla_matrix(ws, anchor_r, anchor_c, product, gender):
    """
    Anchor at (r, c) = title cell.
    Header row at r+1: col c = 'Salary', cols c+1..c+N = year headers.
    Data rows at r+2..: col c = salary_from, cols c+1..c+N = factors.
    Walk salary_from column until blank.
    """
    header_row = anchor_r + 1

    # Discover year columns dynamically (walk right from c+1 until blank)
    year_cols = []
    for dc in range(1, 30):
        v = ws.cell(row=header_row, column=anchor_c + dc).value
        y = as_int(v)
        if y is None:
            break
        year_cols.append((anchor_c + dc, y))

    rows_out = []
    r = header_row + 1
    while r < header_row + 1 + MAX_WALK:
        salary_from = as_int(ws.cell(row=r, column=anchor_c).value)
        if salary_from is None:
            break
        for col, year in year_cols:
            factor = as_float(ws.cell(row=r, column=col).value)
            rows_out.append({
                "product":     product,
                "gender":      gender,
                "salary_from": salary_from,
                "year":        year,
                "factor":      factor,
            })
        r += 1

    return rows_out, len(year_cols), r - header_row - 1  # rows, years, bands


def extract_lsdb_list(ws, anchor_r, anchor_c, product, gender):
    """
    Anchor at (r, c) = 'Year' column header.
    Adjacent col (c+1) = 'Loading' header.
    Data rows at r+1..: col c = year, col c+1 = factor.
    Walk year column until blank.
    """
    rows_out = []
    r = anchor_r + 1
    while r < anchor_r + 1 + MAX_WALK:
        year = as_int(ws.cell(row=r, column=anchor_c).value)
        if year is None:
            break
        factor = as_float(ws.cell(row=r, column=anchor_c + 1).value)
        rows_out.append({
            "product":     product,
            "gender":      gender,
            "salary_from": None,      # LSDB has no band dimension
            "year":        year,
            "factor":      factor,
        })
        r += 1
    return rows_out


# ── Dispatch ───────────────────────────────────────────────────
for spec in SPECS:
    ws = wb[spec["sheet"]]
    hits = find_anchors(ws, spec["anchor_text"])
    ar, ac = hits[spec["anchor_occ"] - 1]

    if spec["shape"] == "gla_matrix":
        rows, n_years, n_bands = extract_gla_matrix(ws, ar, ac, spec["product"], spec["gender"])
        print(f"   {spec['product']}/{spec['gender']:<4} → "
              f"{n_bands} bands × {n_years} years = {len(rows):>3} rows "
              f"(expected {spec['expected_rows']})")
    elif spec["shape"] == "lsdb_list":
        rows = extract_lsdb_list(ws, ar, ac, spec["product"], spec["gender"])
        print(f"   {spec['product']}/{spec['gender']:<4} → "
              f"{len(rows):>3} year rows "
              f"(expected {spec['expected_rows']})")
    else:
        raise ValueError(f"Unknown shape: {spec['shape']}")

    all_rows.extend(rows)

# ── Assemble DataFrame ─────────────────────────────────────────
df = pl.DataFrame(all_rows, infer_schema_length=None)

print(f"\n✅ Cell 4 — Extract complete")
print(f"   Total rows        : {df.height}")
print(f"   Distinct products : {df['product'].n_unique()}")
print(f"   Schema            : {df.schema}")
print("\n── Preview per product/gender ──")
for (p, g), _ in df.group_by(["product", "gender"], maintain_order=True):
    sub = df.filter((pl.col("product") == p) & (pl.col("gender") == g))
    print(f"\n{p}/{g}  ({sub.height} rows):")
    print(sub.head(3))
    print(sub.tail(3))

   gla/M    → 3 bands × 9 years =  27 rows (expected 27)
   gla/F    → 3 bands × 9 years =  27 rows (expected 27)
   lsdb/both →  12 year rows (expected 12)

✅ Cell 4 — Extract complete
   Total rows        : 66
   Distinct products : 2
   Schema            : Schema([('product', String), ('gender', String), ('salary_from', Int64), ('year', Int64), ('factor', Float64)])

── Preview per product/gender ──

gla/M  (27 rows):
shape: (3, 5)
┌─────────┬────────┬─────────────┬──────┬────────┐
│ product ┆ gender ┆ salary_from ┆ year ┆ factor │
│ ---     ┆ ---    ┆ ---         ┆ ---  ┆ ---    │
│ str     ┆ str    ┆ i64         ┆ i64  ┆ f64    │
╞═════════╪════════╪═════════════╪══════╪════════╡
│ gla     ┆ M      ┆ 0           ┆ 2015 ┆ 1.08   │
│ gla     ┆ M      ┆ 0           ┆ 2016 ┆ 1.06   │
│ gla     ┆ M      ┆ 0           ┆ 2017 ┆ 1.04   │
└─────────┴────────┴─────────────┴──────┴────────┘
shape: (3, 5)
┌─────────┬────────┬─────────────┬──────┬────────┐
│ product ┆ gender ┆ salary_from ┆ ye

In [6]:
# ═══════════════════════════════════════════════════════════════
# build_trend.ipynb — Cell 5: Sanity checks
# ═══════════════════════════════════════════════════════════════

# ── Structural checks ──────────────────────────────────────────
assert df.height == 66, f"expected 66 rows, got {df.height}"
assert set(df.columns) == {"product", "gender", "salary_from", "year", "factor"}, \
    f"unexpected columns: {df.columns}"

# ── Per-product/gender row counts ──────────────────────────────
expected_counts = {
    ("gla",  "M"):    27,
    ("gla",  "F"):    27,
    ("lsdb", "both"): 12,
}
actual = {
    (r["product"], r["gender"]): r["len"]
    for r in df.group_by(["product", "gender"]).len().to_dicts()
}
for key, expected in expected_counts.items():
    got = actual.get(key, 0)
    assert got == expected, f"{key}: expected {expected}, got {got}"
    print(f"   ✅ {key[0]}/{key[1]:<4} = {got} rows")

# ── Year ranges ────────────────────────────────────────────────
gla_years = sorted(df.filter(pl.col("product") == "gla")["year"].unique().to_list())
lsdb_years = sorted(df.filter(pl.col("product") == "lsdb")["year"].unique().to_list())
assert gla_years == list(range(2015, 2024)),   f"gla years: {gla_years}"
assert lsdb_years == list(range(2015, 2027)),  f"lsdb years: {lsdb_years}"
print(f"   ✅ gla year range  : {gla_years[0]}..{gla_years[-1]} ({len(gla_years)} years)")
print(f"   ✅ lsdb year range : {lsdb_years[0]}..{lsdb_years[-1]} ({len(lsdb_years)} years)")

# ── Salary band expectations ───────────────────────────────────
gla_m_bands = sorted(df.filter(
    (pl.col("product") == "gla") & (pl.col("gender") == "M")
)["salary_from"].unique().to_list())
gla_f_bands = sorted(df.filter(
    (pl.col("product") == "gla") & (pl.col("gender") == "F")
)["salary_from"].unique().to_list())
assert gla_m_bands == [0, 226000, 504000], f"gla M bands: {gla_m_bands}"
assert gla_f_bands == [0, 214000, 441000], f"gla F bands: {gla_f_bands}"
print(f"   ✅ gla M bands     : {gla_m_bands}")
print(f"   ✅ gla F bands     : {gla_f_bands}")

# LSDB has null salary_from
lsdb_null = df.filter(pl.col("product") == "lsdb")["salary_from"].null_count()
assert lsdb_null == 12, f"lsdb should have 12 null salary_from, got {lsdb_null}"

# ── Value spot-checks (from diagnostic dump) ───────────────────
SPOT_CHECKS = [
    # (product, gender, salary_from, year, expected_factor)
    ("gla",  "M",    0,      2015, 1.08),
    ("gla",  "M",    0,      2019, 1.00),
    ("gla",  "M",    0,      2023, 0.92),
    ("gla",  "M",    226000, 2015, 1.04),
    ("gla",  "M",    504000, 2015, 0.80),
    ("gla",  "M",    504000, 2023, 1.20),
    ("gla",  "F",    0,      2015, 1.10),
    ("gla",  "F",    214000, 2019, 1.00),
    ("gla",  "F",    441000, 2023, 1.04),
    ("lsdb", "both", None,   2015, 1.03),
    ("lsdb", "both", None,   2020, 0.91),  # COVID adjustment
    ("lsdb", "both", None,   2021, 0.91),  # COVID adjustment
    ("lsdb", "both", None,   2026, 1.03),
]

TOL = 1e-6
print("\n🎯 Spot checks:")
fails = 0
for product, gender, salary_from, year, expected in SPOT_CHECKS:
    if salary_from is None:
        sub = df.filter(
            (pl.col("product") == product) &
            (pl.col("gender") == gender) &
            (pl.col("salary_from").is_null()) &
            (pl.col("year") == year)
        )
    else:
        sub = df.filter(
            (pl.col("product") == product) &
            (pl.col("gender") == gender) &
            (pl.col("salary_from") == salary_from) &
            (pl.col("year") == year)
        )
    got = sub["factor"][0] if sub.height else None
    ok = got is not None and abs(got - expected) < TOL
    flag = "✅" if ok else "❌"
    sf_str = "null" if salary_from is None else f"{salary_from}"
    print(f"   {flag}  {product:<5} {gender:<4} sf={sf_str:<6} y={year}  "
          f"expected={expected:<6} got={got}")
    if not ok:
        fails += 1

assert fails == 0, f"{fails} spot check(s) failed"

# ── No null factors ────────────────────────────────────────────
assert df["factor"].null_count() == 0, "found null factor(s)"

# ── Advisory: flag COVID-era LSDB adjustment ───────────────────
covid = df.filter(
    (pl.col("product") == "lsdb") & (pl.col("year").is_in([2020, 2021]))
)
print(f"\n📌 Note: LSDB COVID-era loading (2020, 2021) = "
      f"{covid['factor'].unique().to_list()} (all other years = 1.03)")

print("\n✅ Cell 5 — All sanity checks passed")

   ✅ gla/M    = 27 rows
   ✅ gla/F    = 27 rows
   ✅ lsdb/both = 12 rows
   ✅ gla year range  : 2015..2023 (9 years)
   ✅ lsdb year range : 2015..2026 (12 years)
   ✅ gla M bands     : [0, 226000, 504000]
   ✅ gla F bands     : [0, 214000, 441000]

🎯 Spot checks:
   ✅  gla   M    sf=0      y=2015  expected=1.08   got=1.08
   ✅  gla   M    sf=0      y=2019  expected=1.0    got=1.0
   ✅  gla   M    sf=0      y=2023  expected=0.92   got=0.9199999999999999
   ✅  gla   M    sf=226000 y=2015  expected=1.04   got=1.04
   ✅  gla   M    sf=504000 y=2015  expected=0.8    got=0.7999999999999998
   ✅  gla   M    sf=504000 y=2023  expected=1.2    got=1.2000000000000002
   ✅  gla   F    sf=0      y=2015  expected=1.1    got=1.0999999999999996
   ✅  gla   F    sf=214000 y=2019  expected=1.0    got=1.0
   ✅  gla   F    sf=441000 y=2023  expected=1.04   got=1.04
   ✅  lsdb  both sf=null   y=2015  expected=1.03   got=1.03
   ✅  lsdb  both sf=null   y=2020  expected=0.91   got=0.91
   ✅  lsdb  both sf=nu

In [7]:
# ═══════════════════════════════════════════════════════════════
# build_trend.ipynb — Cell 6: Write outputs + build log
# ═══════════════════════════════════════════════════════════════

# ── 1) Parquet ─────────────────────────────────────────────────
df.write_parquet(PARQUET_PATH)
parquet_hash = sha256_of(PARQUET_PATH)
parquet_kb   = round(PARQUET_PATH.stat().st_size / 1024, 2)
print(f"📦 Parquet written : {PARQUET_PATH.name}  ({parquet_kb} KB)")

# ── 2) Master xlsx ─────────────────────────────────────────────
if MASTER_XLSX.exists():
    try:
        with open(MASTER_XLSX, "a"):
            pass
    except PermissionError:
        raise PermissionError(
            f"'{MASTER_XLSX.name}' is open in Excel. Close it and re-run."
        )

if MASTER_XLSX.exists():
    wb_master = load_workbook(MASTER_XLSX)
else:
    from openpyxl import Workbook
    wb_master = Workbook()
    if "Sheet" in wb_master.sheetnames and len(wb_master.sheetnames) == 1:
        del wb_master["Sheet"]

if SHEET_NAME in wb_master.sheetnames:
    del wb_master[SHEET_NAME]

ws_new = wb_master.create_sheet(SHEET_NAME)

columns = df.columns
ws_new.append(columns)
for row in df.iter_rows():
    ws_new.append(list(row))

# Header row styling
header_row = next(ws_new.iter_rows(min_row=1, max_row=1))
for cell in header_row:
    cell.font      = Font(bold=True, color="FFFFFF")
    cell.fill      = PatternFill("solid", fgColor="305496")
    cell.alignment = Alignment(horizontal="left")

# Column widths
for col_idx, col_name in enumerate(columns, start=1):
    max_len = max(
        [len(str(col_name))] +
        [len(str(v)) if v is not None else 0 for v in df[col_name].to_list()]
    )
    ws_new.column_dimensions[get_column_letter(col_idx)].width = max(12, max_len + 2)

# Excel Table
last_col_letter = get_column_letter(len(columns))
table_ref = f"A1:{last_col_letter}{df.height + 1}"
tbl = Table(displayName=f"tbl_{SHEET_NAME}", ref=table_ref)
tbl.tableStyleInfo = TableStyleInfo(
    name="TableStyleMedium2",
    showFirstColumn=False,
    showLastColumn=False,
    showRowStripes=True,
    showColumnStripes=False,
)
ws_new.add_table(tbl)

wb_master.save(MASTER_XLSX)
print(f"📗 Master xlsx     : {MASTER_XLSX.name} — sheet '{SHEET_NAME}' upserted")

# ── 3) Build log ───────────────────────────────────────────────
source_hash = sha256_of(SOURCE)

log_row = {
    "timestamp_utc":     utc_now_iso(),
    "sheet":             SHEET_NAME,
    "row_count":         df.height,
    "distinct_products": df["product"].n_unique(),
    "source_file":       SOURCE.name,
    "source_sha256":     source_hash,
    "parquet_file":      PARQUET_PATH.name,
    "parquet_sha256":    parquet_hash,
    "parquet_kb":        parquet_kb,
    "status":            "ok",
    "built_by":          BUILT_BY,
}

log_exists = BUILD_LOG.exists()
with open(BUILD_LOG, "a", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=list(log_row.keys()))
    if not log_exists:
        writer.writeheader()
    writer.writerow(log_row)

print(f"📝 Build log       : appended row for '{SHEET_NAME}'")
print(f"   source_sha256   : {source_hash[:16]}…")
print(f"   parquet_sha256  : {parquet_hash[:16]}…")

print(f"\n✅ Cell 6 — trend build COMPLETE")
print(f"   Rows            : {df.height}")
print(f"   Products        : {sorted(df['product'].unique().to_list())}")
print(f"   Parquet         : {PARQUET_PATH}")
print(f"   Master sheet    : {SHEET_NAME} in {MASTER_XLSX.name}")

📦 Parquet written : trend.parquet  (2.13 KB)
📗 Master xlsx     : basis_master_v2026_07.xlsx — sheet 'trend' upserted
📝 Build log       : appended row for 'trend'
   source_sha256   : 10d700d0c785719c…
   parquet_sha256  : b261a18f475a0719…

✅ Cell 6 — trend build COMPLETE
   Rows            : 66
   Products        : ['gla', 'lsdb']
   Parquet         : J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine\basis_snapshots\basis_v2026_07\trend.parquet
   Master sheet    : trend in basis_master_v2026_07.xlsx
